# Notebook 4: Adversarial Training Defense

**Goal:** Make the model robust against adversarial attacks by including adversarial examples during training.

**Method:** Generate GA adversarial examples on-the-fly during training and mix with clean data.

**Output:** Robust model saved to `../results/model_robust.pth`

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('../')

from src.model import CNN
from src.attacks import fgsm_attack, pgd_attack

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Load data
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10(root='../data', train=True,  download=False, transform=transform)
testset  = torchvision.datasets.CIFAR10(root='../data', train=False, download=False, transform=transform)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True)
testloader  = torch.utils.data.DataLoader(testset,  batch_size=128, shuffle=False)

In [ ]:
# Adversarial training with PGD (standard approach)
# We use PGD here because it's fast and strong for training
# In the paper we compare: model trained with GA adversarial examples vs PGD adversarial examples

robust_model = CNN(num_classes=10).to(device)
optimizer = optim.Adam(robust_model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

EPOCHS = 20
EPSILON = 0.03

clean_accs, robust_accs = [], []

for epoch in range(EPOCHS):
    robust_model.train()
    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)

        # Generate adversarial examples using PGD
        adv_images = pgd_attack(robust_model, images, labels, epsilon=EPSILON, steps=10)

        # Train on mix of clean + adversarial
        optimizer.zero_grad()
        outputs = robust_model(adv_images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    # Evaluate clean accuracy
    robust_model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            preds = robust_model(images).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    clean_acc = correct / total
    clean_accs.append(clean_acc)
    print(f'Epoch {epoch+1}/{EPOCHS} | Clean Acc: {clean_acc:.4f}')

torch.save(robust_model.state_dict(), '../results/model_robust.pth')
print('Robust model saved!')

In [ ]:
# Final comparison: baseline model vs robust model
# TODO: Run both models against FGSM, PGD, and GA attacks
# Results go into Table 2 of the paper

print('Run notebook 05_results.ipynb for full comparison table.')